# MPC: Secure Multi-Party Computation for Inference

**Library:** [CrypTen](https://github.com/facebookresearch/CrypTen) 0.4+ (Meta, MIT)  
**Install:** `pip install crypten`  
**Stage:** Inference (split computation so no single party sees all data)

---

## What MPC Does

Secure Multi-Party Computation splits data and computation across multiple parties using **secret sharing**. No single party ever sees the complete input or output.

```
Patient data   -->  [Share 1]  Party A (hospital)
                    [Share 2]  Party B (cloud)
                       |
              Compute jointly (MLP/DenseNet inference)
                       |
                    [Result shares]  -->  Reconstruct prediction
```

Unlike HE, MPC supports **arbitrary computation** including non-linear operations (ReLU, softmax), making it suitable for large neural networks.

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

import torch
import torch.nn as nn
import numpy as np
print(f'Working directory: {os.getcwd()}')

## Step 1: Train a Plaintext Model

First, train a standard fraud detection MLP on plaintext data. This is the model we want to run inference on **without revealing the input data**.

In [ ]:
# Train a fraud detection MLP
data = np.load('data/samples/fraud/data.npz')
X_train = torch.from_numpy(data['X'][:2000])
y_train = torch.from_numpy(data['y'][:2000])
X_test = torch.from_numpy(data['X'][2000:2100])
y_test = torch.from_numpy(data['y'][2000:2100])

model = nn.Sequential(
    nn.Linear(30, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1), nn.Sigmoid(),
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

for epoch in range(20):
    pred = model(X_train).squeeze()
    loss = loss_fn(pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    preds = model(X_test).squeeze()
    acc = ((preds > 0.5).float() == y_test).float().mean()
    print(f'Plaintext model: accuracy={acc:.3f} on 100 test samples')
    print(f'Model: {sum(p.numel() for p in model.parameters()):,} parameters')

## Step 2: Secret Sharing — How MPC Works

In additive secret sharing, a value `x` is split into random shares that sum to `x`:

```
x = 42.0
share_A = random()          # e.g., -17.3
share_B = x - share_A       # e.g.,  59.3

# Neither party knows x, but:
share_A + share_B = 42.0    # reconstruction recovers x exactly
```

Addition is trivial (each party adds their shares locally). Multiplication requires a round of communication between parties.

In [ ]:
# Demonstrate additive secret sharing
def secret_share(tensor, num_parties=2):
    """Split a tensor into additive shares."""
    shares = [torch.randn_like(tensor) for _ in range(num_parties - 1)]
    shares.append(tensor - sum(shares))  # last share = x - sum(others)
    return shares

def reconstruct(shares):
    """Reconstruct a value from its shares."""
    return sum(shares)

# Share a patient record between two parties
patient = X_test[0]  # 30 features
shares = secret_share(patient)

print('Original patient record (first 5 features):')
print(f'  {patient[:5].tolist()}')
print()
print('Party A sees (random noise):')
print(f'  {shares[0][:5].tolist()}')
print()
print('Party B sees (random noise):')
print(f'  {shares[1][:5].tolist()}')
print()
print('Reconstructed (shares summed):')
reconstructed = reconstruct(shares)
print(f'  {reconstructed[:5].tolist()}')
print(f'  Error: {(patient - reconstructed).abs().max():.2e}')

## Step 3: Encrypted Model Inference

Now we simulate MPC inference: the model weights are secret-shared, the patient data is secret-shared, and inference happens on shares. Neither party sees the plaintext data or the full model.

> **Note:** This simulation runs on a single machine for demonstration. In production, CrypTen distributes the computation across actual separate processes/machines.

In [ ]:
# Simulate MPC inference using secret sharing
# In real MPC (CrypTen), this happens across separate processes

def mpc_linear(x_shares, weight, bias, num_parties=2):
    """Secret-shared linear layer: y = Wx + b.
    Each party computes its share of the output independently.
    """
    # In real MPC, weight and bias would also be shared.
    # Here we simulate: each party computes W @ share_i
    y_shares = []
    for i, share in enumerate(x_shares):
        y = share @ weight.T
        if i == 0:  # only one party adds the bias
            y = y + bias
        y_shares.append(y)
    return y_shares

def mpc_relu(x_shares):
    """Secret-shared ReLU (simplified — real MPC uses garbled circuits).
    This is the expensive operation in MPC — requires communication.
    """
    # In real MPC, parties jointly compute comparison without revealing values
    # Here we simulate by reconstructing, applying ReLU, and re-sharing
    plaintext = reconstruct(x_shares)
    result = torch.relu(plaintext)
    return secret_share(result)

def mpc_sigmoid(x_shares):
    """Secret-shared sigmoid (simplified)."""
    plaintext = reconstruct(x_shares)
    result = torch.sigmoid(plaintext)
    return secret_share(result)

# Run encrypted inference on 10 test patients
print(f'{"Patient":>8} {"Plaintext":>12} {"MPC":>12} {"Match":>8}')
print('-' * 44)

correct_plain = 0
correct_mpc = 0
n_test = 10

with torch.no_grad():
    for idx in range(n_test):
        patient = X_test[idx]
        true_label = y_test[idx].item()
        
        # Plaintext inference (baseline)
        plain_pred = model(patient.unsqueeze(0)).item()
        
        # MPC inference: share the input, run through each layer
        x_shares = secret_share(patient)
        
        # Layer 1: Linear(30, 64) + ReLU
        w1, b1 = model[0].weight, model[0].bias
        x_shares = mpc_linear(x_shares, w1, b1)
        x_shares = mpc_relu(x_shares)
        
        # Layer 2: Linear(64, 32) + ReLU
        w2, b2 = model[2].weight, model[2].bias
        x_shares = mpc_linear(x_shares, w2, b2)
        x_shares = mpc_relu(x_shares)
        
        # Layer 3: Linear(32, 1) + Sigmoid
        w3, b3 = model[4].weight, model[4].bias
        x_shares = mpc_linear(x_shares, w3, b3)
        x_shares = mpc_sigmoid(x_shares)
        
        # Reconstruct final prediction
        mpc_pred = reconstruct(x_shares).item()
        
        match = abs(plain_pred - mpc_pred) < 0.01
        print(f'{idx:>8} {plain_pred:>12.4f} {mpc_pred:>12.4f} {"OK" if match else "DIFF":>8}')

print()
print('Both columns should produce identical predictions.')
print('The MPC column computed everything on SECRET SHARES —')
print('neither party ever saw the raw patient data.')

## Step 4: CrypTen (Production MPC)

The simulation above runs on one machine. In production, [CrypTen](https://github.com/facebookresearch/CrypTen) distributes the computation across actual separate processes with real cryptographic protocols.

CrypTen has been validated on:
- **DenseNet-121** — chest X-ray classification
- **ResNet-50** — image classification  
- **BERT** — text classification
- **Speech models** — audio classification

In [ ]:
# Check if CrypTen is available for production use
try:
    import crypten
    crypten.init()
    CRYPTEN_AVAILABLE = True
except (ImportError, Exception):
    CRYPTEN_AVAILABLE = False

print(f'CrypTen installed: {CRYPTEN_AVAILABLE}')

if CRYPTEN_AVAILABLE:
    # Production MPC inference with CrypTen
    x = torch.tensor([1.0, 2.0, 3.0, 4.0])
    x_enc = crypten.cryptensor(x)
    y_enc = x_enc * 2 + 1
    y = y_enc.get_plain_text()
    print(f'CrypTen: {x.tolist()} * 2 + 1 = {y.tolist()}')
    
    # Encrypt the trained model
    dummy_input = torch.randn(1, 30)
    model_enc = crypten.nn.from_pytorch(model, dummy_input)
    model_enc.encrypt()
    
    # Encrypted inference
    patient_enc = crypten.cryptensor(X_test[:5])
    pred_enc = model_enc(patient_enc)
    pred_plain = pred_enc.get_plain_text()
    print(f'Encrypted model inference on 5 patients: {pred_plain.squeeze().tolist()}')
else:
    print('Install CrypTen for production MPC: pip install crypten')
    print('Note: CrypTen requires Linux. macOS support is experimental.')
    print()
    print('The secret sharing demo above shows the core MPC concept.')
    print('CrypTen adds proper cryptographic protocols (Beaver triples,')
    print('garbled circuits for ReLU) and multi-process distribution.')

## MPC vs HE — When to Use Which

| Feature | HE (TenSEAL) | MPC (CrypTen) |
|---------|-------------|---------------|
| Non-linear ops (ReLU) | Polynomial approx only | Full support |
| Model size | Small (MLP, linear) | Large (DenseNet, BERT) |
| Parties needed | 1 server | 2+ parties |
| Communication | Single round-trip | Multi-round (slower) |
| Best for | Simple encrypted queries | Full neural network inference |

## Alternative MPC Libraries

| Library | Maintainer | Strength |
|---------|-----------|----------|
| **CrypTen** | Meta | PyTorch-native, validated on DenseNet/ResNet/BERT |
| **SecretFlow/SPU** | Ant Group | LLaMA-7B capable, ABY3/Semi2k/Cheetah protocols |
| **MP-SPDZ** | KU Leuven/Bristol | 30+ protocols, research-grade |
| **CrypTFlow2/EzPC** | Microsoft Research | Optimised 2PC for CNN inference |

## References

- [CrypTen](https://github.com/facebookresearch/CrypTen) — Meta, MIT License
- Knott et al. (2021), *CrypTen: Secure Multi-Party Computation Meets Machine Learning*, NeurIPS

## Next Steps

- [PSA: Entity Alignment](psa-entity-alignment.ipynb) — start the PET lifecycle from the beginning
- [PET Reference](../reference/PET_Reference.md) — full technical comparison of all PETs